In [1]:
!git clone https://github.com/bremsstrahlung-57/practicum-project.git

Cloning into 'practicum-project'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 55 (delta 8), reused 13 (delta 3), pack-reused 35 (from 1)
Receiving objects: 100% (55/55), 148.11 MiB | 20.82 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Updating files: 100% (12/12), done.


In [2]:
from google.colab import userdata
WANDB_API_KEY = userdata.get('WANDB_API_KEY')
import wandb
wandb.login(WANDB_API_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sagar-sharma-03015611924 (practicum-project) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import torch.nn as nn
import torch
import torch.ao.quantization as tq
import copy
import time
import os
import torchvision
from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

In [4]:
CALIB_BATCHES = 100        # how many batches to run through observer
WARMUP_RUNS   = 20
TIMING_RUNS   = 100
DEVICE        = 'cpu'      # quantized models only run on CPU
PRUNED        = 10

print(torch.backends.quantized.supported_engines)

torch.backends.quantized.engine = (
    'fbgemm' if 'fbgemm' in torch.backends.quantized.supported_engines
    else 'qnnpack'
)

['qnnpack', 'onednn', 'x86', 'fbgemm']


In [5]:
# do not use it
def fuse_resnet(model):
    for m in model.modules():
        if type(m) == torchvision.models.resnet.BasicBlock:
            tq.fuse_modules(m, ['conv1', 'bn1', 'relu'], inplace=True)
            tq.fuse_modules(m, ['conv2', 'bn2'], inplace=True)

In [14]:
def ResNet18CIFAR():
    model = torchvision.models.resnet18(weights=None)
    model.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = torch.nn.Identity()
    model.fc = torch.nn.Linear(512, 10)
    return model

model_fp32 = ResNet18CIFAR()
# model_fp32.load_state_dict(torch.load(f'/content/practicum-project/resnet18_cifar10_baseline.pth', map_location='cpu'))
model_fp32.load_state_dict(torch.load(f'/content/practicum-project/pruned/pruned_{PRUNED}.pth', map_location='cpu'))

model_fp32.eval().to(DEVICE)

model_fused = copy.deepcopy(model_fp32)
# fuse_resnet(model_fused)

qconfig_mapping = get_default_qconfig_mapping("fbgemm")

In [15]:
example_input = torch.randn(1, 3, 32, 32)

model_prepared = prepare_fx(model_fused, qconfig_mapping, example_input)

/tmp/ipykernel_1672/1351101544.py:3: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = prepare_fx(model_fused, qconfig_mapping, example_input)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated i

In [16]:
import torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform_train)
testset  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=True, transform=transform_test)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True,
                         num_workers=2, pin_memory=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=128, shuffle=False,
                         num_workers=2, pin_memory=True)


In [17]:
transform_calib = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

calibset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform_calib
)

calibloader = torch.utils.data.DataLoader(calibset, batch_size=128, shuffle=True)

images, _ = next(iter(calibloader))
print(images.min(), images.max())

tensor(-2.4291) tensor(2.7537)


In [18]:
def calibrate(model, loader, num_batches=CALIB_BATCHES):
    model.eval()
    with torch.no_grad():
        for i, (images, _) in enumerate(loader):
            model(images)
            if i >= num_batches:
                break

calibrate(model_prepared, calibloader)

In [12]:
def evaluate(net, loader, device=DEVICE):
    net.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = net(images)
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

In [20]:
model_int8 = convert_fx(model_prepared)

/tmp/ipykernel_1672/2693613654.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = convert_fx(model_prepared)


In [22]:
acc_fp32 = evaluate(model_fp32, testloader)
acc_int8 = evaluate(model_int8, testloader)

print(acc_fp32, acc_int8)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


95.28


In [23]:
def run_single():
    import tracemalloc

    print(f"FP32 Accuracy : {acc_fp32:.2f}%")
    print(f"INT8 Accuracy : {acc_int8:.2f}%")

    def get_model_size_mb(net, path='tmp.pth'):
        torch.save(net.state_dict(), path)
        size = os.path.getsize(path) / 1e6
        os.remove(path)
        return size

    size_fp32 = get_model_size_mb(model_fp32)
    size_int8 = get_model_size_mb(model_int8)
    print(f"FP32: {size_fp32:.2f} MB  |  INT8: {size_int8:.2f} MB  |  Reduction: {size_fp32/size_int8:.2f}x")

    def benchmark_latency(net, warmup=20, runs=100):
        net.eval()
        dummy = torch.randn(1, 3, 32, 32)
        with torch.no_grad():
            for _ in range(warmup):
                net(dummy)
        times = []
        with torch.no_grad():
            for _ in range(runs):
                t0 = time.perf_counter()
                net(dummy)
                times.append((time.perf_counter() - t0) * 1000)
        return {'mean_ms': sum(times)/len(times), 'min_ms': min(times), 'max_ms': max(times)}

    lat_fp32 = benchmark_latency(model_fp32)
    lat_int8 = benchmark_latency(model_int8)
    print(f"Latency FP32: {lat_fp32['mean_ms']:.3f} ms  |  INT8: {lat_int8['mean_ms']:.3f} ms  |  Speedup: {lat_fp32['mean_ms']/lat_int8['mean_ms']:.2f}x")

    def benchmark_ram(net):
        net.eval()
        dummy = torch.randn(1, 3, 32, 32)
        tracemalloc.start()
        with torch.no_grad():
            net(dummy)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        return peak / 1e6

    ram_fp32 = benchmark_ram(model_fp32)
    ram_int8 = benchmark_ram(model_int8)
    print(f"RAM FP32: {ram_fp32:.2f} MB  |  INT8: {ram_int8:.2f} MB")

    path = f"resnet18_static_int8_{PRUNED}PRUNED.pth"
    torch.save(model_int8.state_dict(), path)

    wandb.init(project="cnn-compression", name="quantization-static-int8-unfused", reinit=True)
    wandb.log({
        "technique":        "static_ptq_int8",
        "pruned":           PRUNED,
        "accuracy":         acc_int8,
        "model_size_mb":    size_int8,
        "latency_mean_ms":  lat_int8['mean_ms'],
        "latency_min_ms":   lat_int8['min_ms'],
        "latency_max_ms":   lat_int8['max_ms'],
        "ram_peak_mb":      ram_int8,
        "size_reduction_x": round(size_fp32 / size_int8, 2),
        "speedup_x":        round(lat_fp32['mean_ms'] / lat_int8['mean_ms'], 2),
        "acc_drop":         round(acc_int8 - 95.34, 2),
    })
    wandb.finish()

    print("\n── Summary ──────────────────────────")
    print(f"Accuracy drop : {acc_int8 - 95.34:+.2f}%")
    print(f"Size reduction: {size_fp32 / size_int8:.2f}x")
    print(f"Speedup       : {lat_fp32['mean_ms'] / lat_int8['mean_ms']:.2f}x")

FP32 Accuracy : 95.00%
INT8 Accuracy : 95.28%
FP32: 44.77 MB  |  INT8: 11.30 MB  |  Reduction: 3.96x
Latency FP32: 27.912 ms  |  INT8: 13.540 ms  |  Speedup: 2.06x
RAM FP32: 0.00 MB  |  INT8: 0.00 MB


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


acc_drop,▁
accuracy,▁
latency_max_ms,▁
latency_mean_ms,▁
latency_min_ms,▁
model_size_mb,▁
pruned,▁
ram_peak_mb,▁
size_reduction_x,▁
speedup_x,▁
acc_drop,-0.06



── Summary ──────────────────────────
Accuracy drop : -0.06%
Size reduction: 3.96x
Speedup       : 2.06x


In [ ]:
def run_whole_pipeline():

    import wandb

    PRUNED_LEVELS = [10, 30, 50, 70, 90]

    results = []

    for PRUNED in PRUNED_LEVELS:

        print(f"\n===== Running for {PRUNED}% pruning =====")

        # Load model
        model_fp32 = ResNet18CIFAR()
        if PRUNED == "Base":
            model_fp32.load_state_dict(
                torch.load(
                    "/content/practicum-project/resnet18_cifar10_baseline.pth",
                    map_location="cpu",
                )
            )
        else:
            model_fp32.load_state_dict(
                torch.load(
                    f"/content/practicum-project/pruned/pruned_{PRUNED}.pth",
                    map_location="cpu",
                )
            )
        model_fp32.eval().to(DEVICE)

        # Prepare quantization (FX)
        model_fused = copy.deepcopy(model_fp32)

        qconfig_mapping = get_default_qconfig_mapping("fbgemm")
        example_input = torch.randn(1, 3, 32, 32)

        model_prepared = prepare_fx(model_fused, qconfig_mapping, example_input)

        # Calibration
        calibrate(model_prepared, calibloader)

        # Convert
        model_int8 = convert_fx(model_prepared)

        # Evaluate
        acc_int8 = evaluate(model_int8, testloader)

        # FP32 baseline (compute once or cache)
        acc_fp32 = evaluate(model_fp32, testloader)

        # Size
        size_fp32 = get_model_size_mb(model_fp32)
        size_int8 = get_model_size_mb(model_int8)

        # Latency
        lat_fp32 = benchmark_latency(model_fp32)
        lat_int8 = benchmark_latency(model_int8)

        # RAM
        ram_fp32 = benchmark_ram(model_fp32)
        ram_int8 = benchmark_ram(model_int8)

        # Save model
        torch.save(model_int8.state_dict(),
                   f"resnet18_int8_{PRUNED}pruned.pth")

        # Store results locally too
        results.append({
            "pruned": PRUNED,
            "acc_fp32": acc_fp32,
            "acc_int8": acc_int8,
            "size_fp32": size_fp32,
            "size_int8": size_int8,
            "lat_fp32": lat_fp32['mean_ms'],
            "lat_int8": lat_int8['mean_ms'],
        })

        # W&B logging
        wandb.init(
            project="cnn-compression",
            name=f"pruned_{PRUNED}_int8",
            group="static_quant_pruning",
            reinit=True,
            config={
                "pruning_percent": PRUNED,
                "quantization": "static_int8_fx",
                "model": "resnet18_cifar10",
            }
        )

        wandb.log({
            "pruned": PRUNED,

            # Accuracy
            "accuracy_fp32": acc_fp32,
            "accuracy_int8": acc_int8,
            "accuracy_drop": acc_int8 - acc_fp32,

            # Size
            "size_fp32_mb": size_fp32,
            "size_int8_mb": size_int8,
            "compression_ratio": size_fp32 / size_int8,

            # Latency
            "latency_fp32_ms": lat_fp32['mean_ms'],
            "latency_int8_ms": lat_int8['mean_ms'],
            "speedup": lat_fp32['mean_ms'] / lat_int8['mean_ms'],

            # RAM
            "ram_fp32_mb": ram_fp32,
            "ram_int8_mb": ram_int8,
        })

        wandb.finish()


===== Running for 10% pruning =====


/tmp/ipykernel_1672/3663532839.py:25: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = prepare_fx(model_fused, qconfig_mapping, example_input)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated 

accuracy_drop,▁
accuracy_fp32,▁
accuracy_int8,▁
compression_ratio,▁
latency_fp32_ms,▁
latency_int8_ms,▁
pruned,▁
ram_fp32_mb,▁
ram_int8_mb,▁
size_fp32_mb,▁
+2,...



===== Running for 30% pruning =====


/tmp/ipykernel_1672/3663532839.py:25: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = prepare_fx(model_fused, qconfig_mapping, example_input)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated 

...

/tmp/ipykernel_1672/3663532839.py:25: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = prepare_fx(model_fused, qconfig_mapping, example_input)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated 

accuracy_drop,▁
accuracy_fp32,▁
accuracy_int8,▁
compression_ratio,▁
latency_fp32_ms,▁
latency_int8_ms,▁
pruned,▁
ram_fp32_mb,▁
ram_int8_mb,▁
size_fp32_mb,▁
+2,...



===== Running for 90% pruning =====


/tmp/ipykernel_1672/3663532839.py:25: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_prepared = prepare_fx(model_fused, qconfig_mapping, example_input)
/usr/local/lib/python3.12/dist-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated 

accuracy_drop,▁
accuracy_fp32,▁
accuracy_int8,▁
compression_ratio,▁
latency_fp32_ms,▁
latency_int8_ms,▁
pruned,▁
ram_fp32_mb,▁
ram_int8_mb,▁
size_fp32_mb,▁
+2,...
